# Structured Reasoning: Kaggle CPU Runner

This notebook runs the experiment on Kaggle **CPU**. A GPU is optional; do not wait for phone verification or GPU access.

**Important:** turn Internet on in Notebook Settings if that option is available, because the notebook must clone GitHub and download the model/dataset. If Internet cannot be enabled, tell me the exact error shown.

Start with the 2-question smoke test. Only run the 10-question pilot after the smoke test completes.

In [ ]:
import os, platform
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
import torch
torch.set_num_threads(2)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CPU threads:', torch.get_num_threads())

In [ ]:
# Clone the latest project files from GitHub
!git clone https://github.com/itsilsil/structured-reasoning-llm.git
%cd /kaggle/working/structured-reasoning-llm

In [ ]:
# Install project dependencies; keep Kaggle's preinstalled PyTorch
!pip install -q "transformers>=4.45.0" "accelerate>=0.34.0" "datasets>=3.0.0" "sentencepiece>=0.2.0" "pandas>=2.2.0" "tqdm>=4.66.0"

## 1. Smoke test — 2 questions (CPU)

This is deliberately small. The model and GSM8K dataset are downloaded on the first run. CPU generation can take several minutes; let the cell finish.

In [ ]:
# CPU-safe smoke test. If a GPU happens to be available, PyTorch may use it automatically.
!OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 python src/experiment.py --sample-size 2 --seed 42 --num-candidates 2 --max-new-tokens 96

In [ ]:
import pandas as pd, json
summary = pd.read_csv('results/summary.csv')
display(summary)
print('\nRun configuration:')
print(json.dumps(json.load(open('results/run_config.json', encoding='utf-8')), indent=2))

## 2. Full pilot — 10 questions

Only run this after the 2-question smoke test finishes without errors. This reruns the experiment with 10 questions and may take substantially longer on CPU.

In [ ]:
# Run the 10-question pilot:
# !OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 python src/experiment.py --sample-size 10 --seed 42 --num-candidates 2 --max-new-tokens 96

# Then inspect the updated summary:
# display(pd.read_csv('results/summary.csv'))

## 3. Export results

Create a ZIP containing the CSV summary, raw JSONL records, and run configuration. Download it from the Kaggle output files.

In [ ]:
import zipfile, os
files = ['results/summary.csv', 'results/experiment_results.jsonl', 'results/run_config.json']
with zipfile.ZipFile('/kaggle/working/experiment_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in files:
        if os.path.exists(path): z.write(path, arcname=os.path.basename(path))
print('Created /kaggle/working/experiment_results.zip')
print('Included:', [p for p in files if os.path.exists(p)])